In [1]:
import requests
import datetime as dt
import time
import pandas as pd
import numpy as np
from collections import deque
import os

#deribit API
clientID = "KQWpIu3V"
clientSecret = "0GPGBoEWRgGGC0gtIxhQl57RJaDOQ8BbgJuTALbTDyY"

In [2]:
#handle error codes
def coinbaseError(errorCode):
    if errorCode==200:
        return
    elif errorCode==400:
        raise Exception("Bad Request — Invalid request format")
    elif errorCode==401:
        raise Exception("Unauthorised - Invalid API Key")
    elif errorCode==403:
        raise Exception("Forbidden - You do not have access to the requested resource")
    elif errorCode==404:
        raise Exception("Not Found")
    elif errorCode==500:
        raise Exception("Internal Server Error - We had a problem with our server")
    elif errorCode==429:
        raise Exception("Too many requests (10 per second)")
    else:
        raise Exception("Status code not recognised")

In [3]:
#get API time so that the requests work if time is required
def getTime():
    urlTime="https://api.exchange.coinbase.com/time"
    timeResponse=requests.get(urlTime)
    coinbaseError(timeResponse.status_code)
    timeJSON=timeResponse.json()
    return timeJSON['epoch']

In [4]:
#get candles data
def getCandles(endTime, startTime, granularity=3600):
    urlCandles="https://api.exchange.coinbase.com/products/BTC-USD/candles"
    paramsCandles={"start": startTime.isoformat() + "Z", "end": endTime.isoformat() + "Z","granularity": granularity}
    candlesResponse=requests.get(urlCandles,params=paramsCandles)
    coinbaseError(candlesResponse.status_code)
    candlesJSON=candlesResponse.json()
    return list(candlesJSON)

In [16]:
BTCtimes=[]
BTCcloseprices=[]
firstTime = 1609459200 # 1-1-2021
currentTime = dt.datetime.now(dt.timezone.utc)
prevTime = currentTime.timestamp()
while currentTime.timestamp() > firstTime:
    startTime = (currentTime - dt.timedelta(hours=300))
    if (startTime.timestamp()<firstTime):
        startTime = dt.datetime.fromtimestamp(firstTime)
    rawData=getCandles(endTime=currentTime,startTime=startTime)
    for i in rawData:
        if (prevTime-i[0] > 3600):
            print("Warning, gap to previous time is " + str(prevTime - i[0]))
            print(dt.datetime.fromtimestamp(prevTime))
            while prevTime > i[0]:
                prevTime-=3600
                BTCtimes.append(dt.datetime.fromtimestamp(prevTime))
                BTCcloseprices.append(i[4])
        BTCtimes.append(dt.datetime.fromtimestamp(i[0]))
        BTCcloseprices.append(i[4])
        prevTime=i[0]
    currentTime = BTCtimes[-1]
    time.sleep(0.1)
BTCtimes.reverse()
BTCcloseprices.reverse()

Warning, gap to previous time is 21600
2025-10-25 22:00:00
Warning, gap to previous time is 14400
2023-03-04 21:00:00


In [47]:
# Helper: classify each timestamp as 'weekend' or 'weekday'
# Weekend = Saturday 8am UTC to Monday 8am UTC
def is_weekend(BTCtime):
    # Convert to UTC if not already timezone-aware
    if BTCtime.tzinfo is None:
        BTCtime = BTCtime.replace(tzinfo=dt.timezone.utc)
    weekday = BTCtime.weekday()  # Monday=0, Sunday=6
    hour = BTCtime.hour

    # Saturday (5) at 08:00 onwards
    if weekday == 5 and hour >= 8:
        return True
    # All of Sunday (6)
    if weekday == 6:
        return True
    # Monday (0) before 08:00
    if weekday == 0 and hour < 8:
        return True
    return False

# Assign flags
day_flags = ['weekend' if is_weekend(BTCtime) else 'weekday' for BTCtime in BTCtimes]

dow_str = {
        0: 'monday',
        1: 'tuesday',
        2: 'wednesday',
        3: 'thursday',
        4: 'friday',
        5: 'saturday',
        6: 'sunday'
    }

dow = [dow_str[(BTCtime - dt.timedelta(hours=8)).weekday()] for BTCtime in BTCtimes]

In [48]:
btc_prices = pd.DataFrame(data={"session_type": day_flags, "dow": dow, "close": BTCcloseprices}, index=BTCtimes)
btc_prices["log_return"] = np.log(btc_prices['close'] / btc_prices['close'].shift(1))
btc_prices.index = pd.to_datetime(btc_prices.index, utc=True)

In [49]:
btc_prices["rv_1d"] = (
   np.sqrt((btc_prices["log_return"] ** 2).rolling(24).mean()) * np.sqrt(365 * 24)
)

In [50]:
btc_prices

,session_type,dow,close,log_return,rv_1d
2021-01-01 00:00:00+00:00,weekday,thursday,29066.58,NaN,NaN
2021-01-01 01:00:00+00:00,weekday,thursday,29487.95,0.014393,NaN
2021-01-01 02:00:00+00:00,weekday,thursday,29249.05,-0.008135,NaN
2021-01-01 03:00:00+00:00,weekday,thursday,29358.30,0.003728,NaN
2021-01-01 04:00:00+00:00,weekday,thursday,29290.52,-0.002311,NaN
...,...,...,...,...,...
2026-04-30 22:00:00+00:00,weekday,thursday,76207.91,-0.003322,0.281224
2026-04-30 23:00:00+00:00,weekday,thursday,76205.36,-0.000033,0.281216
2026-05-01 00:00:00+00:00,weekday,thursday,76305.78,0.001317,0.280114
2026-05-01 01:00:00+00:00,weekday,thursday,76423.74,0.001545,0.256163


In [51]:
os.makedirs("data", exist_ok=True)
btc_prices.to_csv("data/btc_prices.csv", index=True, index_label="session_start")
print(f"Saved btc_prices.csv ({len(btc_prices)} rows)")

Saved btc_prices.csv (46951 rows)


In [52]:
# ── Config ────────────────────────────────────────────────────────────────────

BACKTEST_START      = "2025-01-01"
BACKTEST_END        = "2026-01-01"
DATA_DIR            = "data"
DERIBIT_HISTORY_URL = "https://history.deribit.com/api/v2/public"

# Only look at trades within this many hours of session open for entry pricing.
# Trades outside this window mean the instrument wasn't liquid at session open.
ENTRY_WINDOW_HOURS  = 2.0

os.makedirs(DATA_DIR, exist_ok=True)

In [53]:
# ── Helper: safe GET with retry ───────────────────────────────────────────────

def safe_get(url: str, params: dict, retries: int = 3, delay: float = 2.0) -> dict:
    """GET with retry logic. Returns empty dict on failure."""
    for attempt in range(retries):
        try:
            resp = requests.get(url, params=params, timeout=20)
            if resp.status_code == 429:
                wait = 10 * (attempt + 1)
                print(f"    Rate limited — sleeping {wait}s...")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            return resp.json()
        except requests.RequestException as e:
            print(f"    Request error (attempt {attempt+1}/{retries}): {e}")
            time.sleep(delay)
    return {}

In [54]:
# ── Step 1: Generate session windows ─────────────────────────────────────────

def generate_session_windows(start: str, end: str) -> pd.DataFrame:
    """
    Generates all 24h session windows between start and end.

    Weekend: Saturday 08:00 UTC → Sunday 08:00 UTC
    Weekday: Monday  08:00 UTC → Tuesday 08:00 UTC

    Returns DataFrame with columns:
      session_start (UTC), session_close (UTC), session_type (str), session_dow (str)
    """
    sessions = []
    current  = pd.Timestamp(start, tz="UTC")
    end_ts   = pd.Timestamp(end,   tz="UTC")

    while current < end_ts:
        dow = current.dayofweek  # 0=Mon, 5=Sat

        if current.hour == 8:
            sessions.append({
                "session_start": current,
                "session_close": current + dt.timedelta(hours=24),
                "session_type": ('weekday' if 0 <= dow <= 4 else 'weekend'),
                "session_dow": dow_str[dow]
            })

        current += dt.timedelta(hours=1)

    df        = pd.DataFrame(sessions)
    n_weekend = len(df[df.session_type == "weekend"])
    n_weekday = len(df[df.session_type == "weekday"])
    print(f"Generated {len(df)} session windows "
          f"({n_weekend} weekends, {n_weekday} weekdays)")
    return df

In [56]:
sessions = generate_session_windows(BACKTEST_START, BACKTEST_END)
sessions.to_csv(f"{DATA_DIR}/sessions.csv", index=False)
print(f"Saved sessions.csv ({len(sessions)} rows)")

Generated 365 session windows (104 weekends, 261 weekdays)
Saved sessions.csv (365 rows)


In [57]:
# ── Step 2: Fetch BTC options trades for a time window ───────────────────────

def fetch_trades(start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    """
    Fetches all BTC options trades between start and end from Deribit.
    Paginates automatically if > 1000 trades.

    Returns DataFrame with columns:
      timestamp, instrument_name, price, iv, amount,
      direction, index_price, underlying_price
    """
    start_ms      = int(start.timestamp() * 1000)
    end_ms        = int(end.timestamp() * 1000)
    all_trades    = []
    current_start = start_ms

    while True:
        params = {
            "currency":        "BTC",
            "kind":            "option",
            "start_timestamp": current_start,
            "end_timestamp":   end_ms,
            "count":           1000,
            "include_old":     "true"
        }
        data   = safe_get(
            f"{DERIBIT_HISTORY_URL}/get_last_trades_by_currency_and_time",
            params
        )
        result = data.get("result", {})
        trades = result.get("trades", [])

        if not trades:
            break

        all_trades.extend(trades)

        if len(trades) < 1000:
            break

        last_ts       = trades[-1]["timestamp"]
        current_start = last_ts + 1

        if current_start >= end_ms:
            break

        time.sleep(0.2)

    if not all_trades:
        return pd.DataFrame()

    df = pd.DataFrame(all_trades)
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)

    keep = [c for c in [
        "timestamp", "instrument_name", "price", "iv",
        "amount", "direction", "index_price", "underlying_price"
    ] if c in df.columns]

    return df[keep].sort_values("timestamp").reset_index(drop=True)

In [59]:
# ── Step 3: Parse instrument name ─────────────────────────────────────────────

def parse_instrument(name: str) -> dict:
    """
    Parses a Deribit instrument name e.g. 'BTC-28MAR25-80000-C'.
    Returns dict: strike (float), expiry (Timestamp), option_type ('C'/'P')
    Returns NaNs on failure.
    """
    try:
        parts       = name.split("-")
        expiry_str  = parts[1] + "-0800"
        strike      = float(parts[2])
        option_type = parts[3]
        expiry      = pd.to_datetime(expiry_str, format="%d%b%y%z", utc=True)
        return {"strike": strike, "expiry": expiry, "option_type": option_type}
    except Exception:
        return {"strike": np.nan, "expiry": pd.NaT, "option_type": None}

In [60]:
# ── Step 4: Identify the 4 trade legs ────────────────────────────────────────

def identify_legs(entry_trades_df: pd.DataFrame,
                  spot:            float,
                  session_close:   pd.Timestamp) -> pd.DataFrame:
    """
    From trades in the ENTRY WINDOW ONLY (first 2h of session), identifies
    the 4 strategy legs:
      atm_call — call with strike closest to spot
      atm_put  — put with strike closest to spot
      otm_call — call with strike closest to spot * 1.15
      otm_put  — put with strike closest to spot * 0.85

    Only uses instruments expiring at or after session_close.
    Picks the nearest available expiry (should be session_close itself
    for daily expiries).

    Using entry_trades_df (not full window) guarantees that any leg
    identified here was actually trading at session open.

    Returns DataFrame with columns:
      leg, instrument_name, strike, option_type, expiry
    """
    if entry_trades_df.empty:
        return pd.DataFrame()

    unique_names = entry_trades_df["instrument_name"].unique()
    parsed = []
    for name in unique_names:
        meta = parse_instrument(name)
        meta["instrument_name"] = name
        parsed.append(meta)

    catalog = pd.DataFrame(parsed).dropna(subset=["strike", "expiry"])

    # Only instruments expiring at or after session close
    catalog = catalog[catalog["expiry"].dt.date >= session_close.date()]

    if catalog.empty:
        return pd.DataFrame()

    # Use nearest expiry — for daily expiries this should == session_close
    # Pick expiry closest to session_close that has actual trades.
    # Proximity is primary key (Mon->Tue picks Tue over Fri),
    # trade count is tiebreaker.
    expiry_counts = {}
    for name in entry_trades_df["instrument_name"].unique():
        parsed = parse_instrument(name)
        exp    = parsed["expiry"]
        if pd.isna(exp):
            continue
        if exp.date() < session_close.date():
            continue
        count = int((entry_trades_df["instrument_name"] == name).sum())
        expiry_counts[exp] = expiry_counts.get(exp, 0) + count

    if not expiry_counts:
        return pd.DataFrame()

    best_expiry = min(
        expiry_counts.keys(),
        key=lambda e: (
            abs((e.date() - session_close.date()).days),
            -expiry_counts[e]
        )
    )
    catalog = catalog[catalog["expiry"] == best_expiry]
    targets = {
        "atm_call": ("C", spot),
        "atm_put":  ("P", spot),
        "otm_call": ("C", spot * 1.15),
        "otm_put":  ("P", spot * 0.85),
    }

    legs = []
    for leg_name, (opt_type, target_strike) in targets.items():
        subset = catalog[catalog["option_type"] == opt_type]
        if subset.empty:
            continue
        idx     = (subset["strike"] - target_strike).abs().idxmin()
        matched = subset.loc[idx].copy()
        matched["leg"] = leg_name
        legs.append(matched)

    if not legs:
        return pd.DataFrame()

    return pd.DataFrame(legs)[
        ["leg", "instrument_name", "strike", "option_type", "expiry"]
    ].reset_index(drop=True)

In [64]:
# ── Step 5: Extract entry price per leg ──────────────────────────────────────

def extract_entry_prices(entry_trades_df: pd.DataFrame,
                         legs_df:         pd.DataFrame,
                         session_start:   pd.Timestamp) -> pd.DataFrame:
    """
    For each leg, finds the trade closest to session_start within the
    entry window trades.

    New columns added to legs_df:
      entry_price      — option price in BTC at entry
      entry_iv         — implied vol at entry (annualised decimal e.g. 0.85)
      entry_underlying — BTC spot price recorded at time of entry trade
      entry_gap_hours  — hours between session_start and nearest entry trade
      data_quality_ok  — True if entry_gap_hours <= ENTRY_WINDOW_HOURS (2h)

    NOTE: Exit price = intrinsic value at expiry, computed in Subtask 7.
      Call intrinsic: max(spot_at_expiry - strike, 0)
      Put intrinsic:  max(strike - spot_at_expiry, 0)
    """

    if entry_trades_df.empty or legs_df.empty:
        return legs_df

    results = []
    for _, leg in legs_df.iterrows():
        inst   = leg["instrument_name"]
        subset = entry_trades_df[
            entry_trades_df["instrument_name"] == inst
        ].copy()
        print(f"    DEBUG {inst}: {len(subset)} trades found") 

        if subset.empty:
            # Instrument was in catalog but no trade found in entry window
            print(f"    ⚠ No entry window trades for {inst} — flagged")
            results.append({
                "entry_price":      np.nan,
                "entry_iv":         np.nan,
                "entry_underlying": np.nan,
                "entry_gap_hours":  np.nan,
                "data_quality_ok":  False,
            })
            continue

        # Trade nearest to session_start
        subset = subset.copy()
        subset["dist"]  = (subset["timestamp"] - session_start).abs()
        entry_row       = subset.loc[subset["dist"].idxmin()]
        entry_gap_hours = entry_row["dist"].total_seconds() / 3600
        data_quality_ok = entry_gap_hours <= ENTRY_WINDOW_HOURS

        if not data_quality_ok:
            print(f"    ⚠ {inst}: nearest trade {entry_gap_hours:.1f}h "
                  f"from open — flagged")
        
        results.append({
        "entry_price":      entry_row["price"]            if "price"            in entry_row.index else np.nan,
        "entry_iv":         entry_row["iv"]/100           if "iv"               in entry_row.index else np.nan,
        "entry_underlying": entry_row["index_price"]      if "index_price"      in entry_row.index else np.nan,
        "entry_gap_hours":  round(entry_gap_hours, 2),
        "data_quality_ok":  data_quality_ok,})
        print(f"    DEBUG entry_row type: {type(entry_row)}")
        print(f"    DEBUG entry_row:\n{entry_row}")
        print(f"    DEBUG price value: {entry_row['price']}")
        

    price_df = pd.DataFrame(results, index=legs_df.index)
    return pd.concat(
        [legs_df.reset_index(drop=True), price_df.reset_index(drop=True)],
        axis=1
    )

In [62]:
# ── Step 6: Main collection loop ──────────────────────────────────────────────

def collect_options_data(sessions:   pd.DataFrame,
                         btc_prices: pd.DataFrame) -> pd.DataFrame:
    """
    Main loop over all session windows.

    For each session:
      1. Gets BTC spot at session open from btc_prices
      2. Fetches trades from session_start to session_start + 2h (entry window)
      3. Identifies 4 legs from instruments trading in that entry window
      4. Extracts entry prices and IVs

    Returns master options_df with one row per leg per session.
    """
    all_legs = []

    for i, session in sessions.iterrows():
        s_start = session["session_start"]
        s_close = session["session_close"]
        s_type  = session["session_type"]
        s_dow   = session["session_dow"] 

        # Get BTC spot at session open
        try:
            spot = float(btc_prices.loc[s_start, "close"])
        except KeyError:
            idx  = btc_prices.index.get_indexer([s_start], method="nearest")[0]
            spot = float(btc_prices.iloc[idx]["close"])

        print(f"\n[{i+1}/{len(sessions)}] {s_type.upper()} | {s_dow.upper()} | "
              f"{s_start.strftime('%Y-%m-%d %H:%M')} → "
              f"{s_close.strftime('%Y-%m-%d %H:%M')} | "
              f"BTC=${spot:,.0f}")

        # Fetch trades in entry window only (session_start → +2h)
        entry_end     = s_start + dt.timedelta(hours=ENTRY_WINDOW_HOURS)
        entry_trades  = fetch_trades(s_start, entry_end)

        if entry_trades.empty:
            print("  No trades in entry window — skipping")
            continue

        print(f"  Entry window: {len(entry_trades)} trades | "
              f"{entry_trades['instrument_name'].nunique()} instruments")

        # Identify the 4 legs from entry window instruments only
        legs_df = identify_legs(entry_trades, spot, s_close)

        if legs_df.empty:
            print("  No legs found expiring at session close — skipping")
            print("  (daily expiry may not have been listed yet at session open)")
            continue

        print(f"  Legs identified:")
        for _, leg in legs_df.iterrows():
            print(f"    {leg['leg']:12s} {leg['instrument_name']:35s} "
                  f"strike=${leg['strike']:,.0f}  "
                  f"expiry={leg['expiry'].strftime('%Y-%m-%d %H:%M UTC')}")

        # Extract entry prices
        legs_df = extract_entry_prices(entry_trades, legs_df, s_start)

        # Attach session metadata
        legs_df["session_start"] = s_start
        legs_df["session_close"] = s_close
        legs_df["session_type"]  = s_type
        legs_df["session_dow"]   = s_dow 
        legs_df["spot_at_open"]  = spot

        all_legs.append(legs_df)
        time.sleep(0.5)

    if not all_legs:
        print("\nWARNING: No data collected across any session")
        return pd.DataFrame()
    print(f"    DEBUG all_legs length: {len(all_legs)}")
    options_df = pd.concat(all_legs, ignore_index=True)

    # Drop legs with no entry price at all
    before     = len(options_df)
    options_df = options_df.dropna(subset=["entry_price"])
    dropped    = before - len(options_df)
    if dropped:
        print(f"\nDropped {dropped} legs with no entry price")

    # Drop IV data errors (> 500% annualised is clearly wrong)
    if "entry_iv" in options_df.columns:
        options_df = options_df[
            options_df["entry_iv"].isna() | (options_df["entry_iv"] <= 5.0)
        ]

    options_df = options_df.sort_values(
        ["session_start", "leg"]
    ).reset_index(drop=True)

    return options_df

In [63]:
# ── Main ──────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("Subtask 1B: BTC Options Data Collection")
    print(f"Period  : {BACKTEST_START} to {BACKTEST_END}")
    print(f"Source  : Deribit public API (no API key required)")
    print(f"Weekend : Sat 08:00 UTC → Sun 08:00 UTC (24h)")
    print(f"Weekday : Mon 08:00 UTC → Tue 08:00 UTC (24h)")
    print(f"Entry window: first {ENTRY_WINDOW_HOURS}h of each session\n")

    # 1. Generate session windows
    sessions = generate_session_windows(BACKTEST_START, BACKTEST_END)
    sessions.to_csv(f"{DATA_DIR}/sessions.csv", index=False)
    print(f"Saved sessions.csv ({len(sessions)} rows)\n")

    # 2. Load BTC spot prices (from existing dataCollection.py)
    btc_prices = pd.read_csv(
        f"{DATA_DIR}/btc_prices.csv", index_col=0, parse_dates=True
    )
    btc_prices.index = pd.to_datetime(btc_prices.index, utc=True)
    btc_prices = btc_prices[~btc_prices.index.duplicated(keep="first")]
    print(f"Loaded btc_prices.csv ({len(btc_prices)} rows)")

    # 3. Collect options data
    options_df = collect_options_data(sessions, btc_prices)

    if options_df.empty:
        print("\nNo data collected — check logs above")
    else:
        total       = len(options_df)
        quality_ok  = int(options_df["data_quality_ok"].sum())
        quality_bad = total - quality_ok

        print(f"\n{'='*55}")
        print(f"COLLECTION COMPLETE")
        print(f"{'='*55}")
        print(f"Sessions collected : {options_df['session_start'].nunique()}")
        print(f"Total legs         : {total}")
        print(f"Quality OK (<2h)   : {quality_ok}  ({100*quality_ok/total:.1f}%)")
        print(f"Flagged (>2h gap)  : {quality_bad} ({100*quality_bad/total:.1f}%)")
        print(f"\nIn downstream subtasks, filter with:")
        print(f"  options_df = options_df[options_df['data_quality_ok']]")
        print(f"{'='*55}")

        options_df.to_csv(f"{DATA_DIR}/options_df.csv", index=False)
        print(f"\nSaved options_df.csv ({len(options_df)} rows)")
        print(f"Columns: {list(options_df.columns)}")

    print("\nDone.")

Subtask 1B: BTC Options Data Collection
Period  : 2025-01-01 to 2026-01-01
Source  : Deribit public API (no API key required)
Weekend : Sat 08:00 UTC → Sun 08:00 UTC (24h)
Weekday : Mon 08:00 UTC → Tue 08:00 UTC (24h)
Entry window: first 2.0h of each session

Generated 365 session windows (104 weekends, 261 weekdays)
Saved sessions.csv (365 rows)

Loaded btc_prices.csv (46701 rows)

[1/365] WEEKDAY | WEDNESDAY | 2025-01-01 08:00 → 2025-01-02 08:00 | BTC=$93,247
  Entry window: 1999 trades | 187 instruments
  Legs identified:
    atm_call     BTC-2JAN25-93000-C                  strike=$93,000  expiry=2025-01-02 08:00 UTC
    atm_put      BTC-2JAN25-93000-P                  strike=$93,000  expiry=2025-01-02 08:00 UTC
    otm_call     BTC-2JAN25-102000-C                 strike=$102,000  expiry=2025-01-02 08:00 UTC
    otm_put      BTC-2JAN25-84000-P                  strike=$84,000  expiry=2025-01-02 08:00 UTC
    DEBUG BTC-2JAN25-93000-C: 14 trades found
Index(['timestamp', 'instrument_na

In [127]:
session_signals = pd.read_csv("data/sessions.csv")
options_df = pd.read_csv("data/options_df.csv")

session_start=session_signals["session_start"].to_list()
session_type=session_signals["session_type"].to_list()
session_dow=session_signals["session_dow"].to_list()
iv=options_df["entry_iv"].to_list()
atm_iv=[]

# Options considered loop through: ATM call, ATM put, OTM call, OTM put 
# so 4i and 4i + 1 are ATM indices
# The IVs should be nearby (put-call parity)
for i in range(len(session_start)):
    atm_iv.append((iv[4*i]+iv[4*i+1])/2)

# IV Rank
# For each row, compute iv_52w_low and iv_52w_high as the min/max of atm_iv
# over the trailing 52 weeks of same-session-type rows.
# Weekend IVR benchmarks against past weekend IVs only, not weekday.

session_signals["session_start"] = pd.to_datetime(session_signals["session_start"], utc=True)
session_signals["session_close"] = pd.to_datetime(session_signals["session_close"], utc=True)
session_signals["atm_iv"] = atm_iv

session_signals = session_signals.sort_values("session_start")

session_signals["iv_52w_low"] = pd.NA
session_signals["iv_52w_high"] = pd.NA
session_signals["iv_rank"] = pd.NA

for stype in session_signals["session_type"].unique():
    mask = session_signals["session_type"] == stype
    subset = session_signals.loc[mask].copy()
    subset = subset.set_index("session_start").sort_index()

    stats = subset["atm_iv"].rolling("364D", min_periods=1).agg(["min", "max"])
    stats.columns = ["iv_52w_low", "iv_52w_high"]

    session_signals.loc[mask, ["iv_52w_low", "iv_52w_high"]] = stats[["iv_52w_low", "iv_52w_high"]].to_numpy()

# EPS x1 in num and x2 in denom to make default value 50%
EPS = 1e-6
session_signals["iv_rank"] = (session_signals["atm_iv"] - session_signals["iv_52w_low"] + EPS) / \
    (session_signals["iv_52w_high"] - session_signals["iv_52w_low"] + 2 * EPS) * 100.

# VRP (atm_iv - rv_at_open)
# RV at open comes from btc_prices.csv, want to merge using some sort of btc_prices.loc[time]
btc_prices = pd.read_csv("data/btc_prices.csv")
btc_prices["session_start"] = pd.to_datetime(btc_prices["session_start"], utc=True)

# Merge rv_5d from btc_prices into session_signals based on matching session_start
session_signals = session_signals.merge(btc_prices[["session_start", "rv_1d"]], on="session_start", how="left")

session_signals["rv_1d"] = session_signals["rv_1d"].shift(-1)

session_signals["vrp"] = session_signals["atm_iv"] - session_signals["rv_1d"]

# Trade condition 1: VRP > THRESHOLD
THRESHOLD = 0.05
session_signals['signal'] = (session_signals['vrp'] > THRESHOLD) & (session_signals['iv_rank'] > 40)

#session_signals['signal'] = session_signals['session_dow'] == 'sunday'

# write results back (at end)
session_signals.to_csv("data/session_signals.csv", index=False)

In [131]:
dt.datetime.timestamp(dt.datetime(2021,1,1))

1609459200.0

In [128]:
session_signals

,session_start,session_close,session_type,session_dow,atm_iv,iv_52w_low,iv_52w_high,iv_rank,rv_1d,vrp,signal
0,2025-01-01 08:00:00+00:00,2025-01-02 08:00:00+00:00,weekday,wednesday,0.43040,0.4304,0.4304,50.0,0.278075,0.152325,True
1,2025-01-02 08:00:00+00:00,2025-01-03 08:00:00+00:00,weekday,thursday,0.53770,0.4304,0.5377,99.999068,0.369986,0.167714,True
2,2025-01-03 08:00:00+00:00,2025-01-04 08:00:00+00:00,weekday,friday,0.40305,0.40305,0.5377,0.000743,0.254323,0.148727,False
3,2025-01-04 08:00:00+00:00,2025-01-05 08:00:00+00:00,weekend,saturday,0.20875,0.20875,0.20875,50.0,0.185265,0.023485,False
4,2025-01-05 08:00:00+00:00,2025-01-06 08:00:00+00:00,weekend,sunday,0.28990,0.20875,0.2899,99.998768,0.208302,0.081598,True
...,...,...,...,...,...,...,...,...,...,...,...
361,2025-12-27 08:00:00+00:00,2025-12-28 08:00:00+00:00,weekend,saturday,0.13110,0.10905,0.53915,5.126923,0.093077,0.038023,False
362,2025-12-28 08:00:00+00:00,2025-12-29 08:00:00+00:00,weekend,sunday,0.32900,0.10905,0.53915,51.139265,0.287797,0.041203,False
363,2025-12-29 08:00:00+00:00,2025-12-30 08:00:00+00:00,weekday,monday,0.36790,0.2206,1.08285,17.083289,0.370196,-0.002296,False
364,2025-12-30 08:00:00+00:00,2025-12-31 08:00:00+00:00,weekday,tuesday,0.36745,0.2206,1.08285,17.0311,0.264228,0.103222,False


In [129]:
session_signals.groupby(['session_dow'])[['atm_iv','rv_1d','vrp','iv_rank']].mean()

,atm_iv,rv_1d,vrp,iv_rank
session_dow,,,,
friday,0.429297,0.405905,0.023392,21.619053
monday,0.443449,0.462804,-0.019355,23.51788
saturday,0.206484,0.202460,0.004024,18.729177
sunday,0.336311,0.413920,-0.077609,53.893155
thursday,0.410862,0.452021,-0.041159,19.812203
tuesday,0.422448,0.418057,0.004391,20.237188
wednesday,0.427185,0.414705,0.013931,22.262862


In [130]:
session_signals[session_signals['signal']]

,session_start,session_close,session_type,session_dow,atm_iv,iv_52w_low,iv_52w_high,iv_rank,rv_1d,vrp,signal
0,2025-01-01 08:00:00+00:00,2025-01-02 08:00:00+00:00,weekday,wednesday,0.43040,0.4304,0.4304,50.0,0.278075,0.152325,True
1,2025-01-02 08:00:00+00:00,2025-01-03 08:00:00+00:00,weekday,thursday,0.53770,0.4304,0.5377,99.999068,0.369986,0.167714,True
4,2025-01-05 08:00:00+00:00,2025-01-06 08:00:00+00:00,weekend,sunday,0.28990,0.20875,0.2899,99.998768,0.208302,0.081598,True
10,2025-01-11 08:00:00+00:00,2025-01-12 08:00:00+00:00,weekend,saturday,0.25275,0.20875,0.2899,54.220475,0.172449,0.080301,True
13,2025-01-14 08:00:00+00:00,2025-01-15 08:00:00+00:00,weekday,tuesday,0.59285,0.40305,0.59285,99.999473,0.478773,0.114077,True
14,2025-01-15 08:00:00+00:00,2025-01-16 08:00:00+00:00,weekday,wednesday,0.61780,0.40305,0.6178,99.999534,0.557865,0.059935,True
20,2025-01-21 08:00:00+00:00,2025-01-22 08:00:00+00:00,weekday,tuesday,0.69275,0.40305,1.08285,42.615497,0.532713,0.160037,True
53,2025-02-23 08:00:00+00:00,2025-02-24 08:00:00+00:00,weekend,sunday,0.35780,0.1616,0.5078,56.672405,0.225334,0.132466,True
58,2025-02-28 08:00:00+00:00,2025-03-01 08:00:00+00:00,weekday,friday,0.88825,0.3716,1.08285,72.639655,0.784144,0.104106,True
63,2025-03-05 08:00:00+00:00,2025-03-06 08:00:00+00:00,weekday,wednesday,0.73245,0.3716,1.08285,50.73462,0.667620,0.064830,True
